In [0]:
df_sales = spark.table("`e-commerce_sales_catalog`.silver.silver_sales_detail")

### KPI 1: Daily Revenue

In [0]:
from pyspark.sql.functions import *

df_daily_revenue = df_sales.withColumn(
    "order_date",
    to_date(col("order_purchase_timestamp"))
).groupBy("order_date").agg(
    sum("total_amount").alias("daily_revenue")
)

In [0]:
df_daily_revenue.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("`e-commerce_sales_catalog`.gold.gold_daily_revenue")

### KPI 2: Top Customers per city

In [0]:
from pyspark.sql.functions import sum

df_top_customers = df_sales.groupBy(
    "customer_id", "customer_city"
).agg(
    sum("total_amount").alias("total_spent")
)

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import rank, col

df_top_customers = df_top_customers.repartition("customer_city")

window_spec = Window.partitionBy("customer_city") \
                    .orderBy(col("total_spent").desc())

df_top_customers = df_top_customers.withColumn(
    "rank",
    rank().over(window_spec)
)

In [0]:
df_top_customers.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("`e-commerce_sales_catalog`.gold.gold_top_customers")

### KPI 3: Top Products

In [0]:
df_top_products = df_sales.groupBy(
    "product_id", "product_category_name"
).agg(
    sum("total_amount").alias("revenue")
)

In [0]:
df_top_products.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("`e-commerce_sales_catalog`.gold.gold_top_products")

### KPI 4: Category Performance

In [0]:
df_category = df_sales.groupBy(
    "product_category_name"
).agg(
    sum("total_amount").alias("total_revenue")
)

In [0]:
df_category.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("`e-commerce_sales_catalog`.gold.gold_category_performance")

### KPI 5: Monthly Revenue Trend

In [0]:
from pyspark.sql.functions import *

df_monthly = df_sales.withColumn(
    "year", year(col("order_purchase_timestamp"))
).withColumn(
    "month", month(col("order_purchase_timestamp"))
).groupBy("year", "month").agg(
    round(sum("total_amount"), 2).alias("monthly_revenue")
)

In [0]:
df_monthly.show()

+----+-----+---------------+
|year|month|monthly_revenue|
+----+-----+---------------+
|2017|    9|      720398.91|
|2018|    3|     1155126.82|
|2017|   12|      863547.23|
|2017|    7|      584971.62|
|2018|    6|     1022677.11|
|2017|   11|     1179143.77|
|2018|    2|      986908.96|
|2017|    4|      412422.24|
|2018|    8|     1003308.47|
|2018|    7|     1058728.03|
|2018|    1|     1107301.89|
|2018|    4|     1159698.04|
|2017|    8|       668204.6|
|2017|    5|      586190.95|
|2017|   10|      769312.37|
|2018|    5|     1149781.82|
|2017|    6|      502963.04|
|2017|    1|      137188.49|
|2017|    3|      432048.59|
|2017|    2|      286280.62|
+----+-----+---------------+
only showing top 20 rows


In [0]:
df_monthly = df_monthly.orderBy("year", "month")

In [0]:
df_monthly.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("`e-commerce_sales_catalog`.gold.gold_monthly_revenue")

In [0]:
spark.table("`e-commerce_sales_catalog`.gold.gold_monthly_revenue").show()

+----+-----+---------------+
|year|month|monthly_revenue|
+----+-----+---------------+
|2016|    9|         354.75|
|2016|   10|       56808.84|
|2016|   12|          19.62|
|2017|    1|      137188.49|
|2017|    2|      286280.62|
|2017|    3|      432048.59|
|2017|    4|      412422.24|
|2017|    5|      586190.95|
|2017|    6|      502963.04|
|2017|    7|      584971.62|
|2017|    8|       668204.6|
|2017|    9|      720398.91|
|2017|   10|      769312.37|
|2017|   11|     1179143.77|
|2017|   12|      863547.23|
|2018|    1|     1107301.89|
|2018|    2|      986908.96|
|2018|    3|     1155126.82|
|2018|    4|     1159698.04|
|2018|    5|     1149781.82|
+----+-----+---------------+
only showing top 20 rows


### KPI 6: Delivery Time Analysis

In [0]:
from pyspark.sql.functions import *

df_delivery = df_sales.withColumn(
    "delivery_days",
    datediff(col("order_delivered_customer_date"), col("order_purchase_timestamp"))
)

In [0]:
df_delivery_kpi = df_delivery.groupBy().agg(
    round(avg("delivery_days"), 2).alias("avg_delivery_days")
)

In [0]:
df_delivery_kpi.show()

+-----------------+
|avg_delivery_days|
+-----------------+
|            12.41|
+-----------------+

